# Naive RAG Walkthrough
투자 트랜스크립트 -> pgvector -> gpt-4o-mini.

## 0. 원본 데이터 수집 파이프라인 (참고)

현재 pgvector에 들어있는 트랜스크립트는 메인 프로젝트(`iterate-to-wealth`, 비공개 repo)가 미리 적재한 것이다. 어떻게 만들어졌는지 알 수 있도록 핵심 흐름을 한 셀에 정리한다.

**파이프라인**: YouTube URL → `yt-dlp`로 m4a 오디오 다운로드 → (선택) `ffmpeg`로 2배속 → OpenAI Whisper API로 한국어 transcribe → PostgreSQL `contents.transcript`에 저장.

아래 셀은 실제 실행을 막아두었다(스터디 셋업 시 다시 돌릴 필요 없음). 이미 DB에 데이터가 들어있고, 새로 돌리려면 `yt-dlp`, `ffmpeg`, OpenAI Whisper 크레딧이 필요하다.

In [ ]:
# 참고용 — 실제 실행 안 함 (DB에 이미 적재됨)
# 실행하려면: `uv pip install yt-dlp openai` + ffmpeg 설치 후 RUN_INGEST = True

RUN_INGEST = False

from pathlib import Path
import subprocess
from datetime import datetime


def download_audio(url: str, data_dir: Path) -> tuple[dict, Path]:
    """yt-dlp로 비디오에서 m4a 오디오만 추출."""
    import yt_dlp

    data_dir.mkdir(parents=True, exist_ok=True)
    opts = {
        "format": "bestaudio/best",
        "outtmpl": str(data_dir / "%(id)s.%(ext)s"),
        "postprocessors": [
            {"key": "FFmpegExtractAudio", "preferredcodec": "m4a"},
        ],
        "quiet": True,
        "no_warnings": True,
    }
    with yt_dlp.YoutubeDL(opts) as ydl:
        info = ydl.extract_info(url, download=True)
    audio_path = next(data_dir.glob(f"{info['id']}.*"))
    return info, audio_path


def speed_up_audio(source: Path, factor: float = 2.0) -> Path:
    """ffmpeg atempo로 N배속 가속 (Whisper 비용 절감)."""
    if factor == 1.0:
        return source
    output = source.with_name(f"{source.stem}.x{factor}{source.suffix}")
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", str(source),
         "-filter:a", f"atempo={factor}", "-vn", str(output)],
        check=True,
    )
    return output


def transcribe(audio_path: Path, api_key: str) -> str:
    """OpenAI Whisper API로 한국어 transcribe."""
    from openai import OpenAI

    client = OpenAI(api_key=api_key)
    with audio_path.open("rb") as f:
        text = client.audio.transcriptions.create(
            model="whisper-1",
            response_format="text",
            file=f,
            language="ko",
        )
    return text.strip() if isinstance(text, str) else str(text).strip()


def save_to_postgres(
    database_url: str,
    *,
    persona_id: str,
    video_id: str,
    url: str,
    title: str,
    published_at: datetime,
    transcript_text: str,
) -> None:
    """contents 테이블에 INSERT. 메인 프로젝트 schema 준수."""
    from sqlalchemy import create_engine, text

    engine = create_engine(database_url)
    with engine.begin() as conn:
        conn.execute(
            text("""
                INSERT INTO contents
                  (id, persona_id, source_type, source_url, title,
                   published_at, transcript, ingested_at)
                VALUES
                  (gen_random_uuid(), :persona_id, 'youtube', :url, :title,
                   :published_at, :transcript, NOW())
                ON CONFLICT (source_url) DO NOTHING
            """),
            {
                "persona_id": persona_id,
                "url": url,
                "title": title,
                "published_at": published_at,
                "transcript": transcript_text,
            },
        )


if RUN_INGEST:
    from naive_rag.config import Settings

    settings = Settings()
    url = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"
    data_dir = Path("data/audio")

    info, audio = download_audio(url, data_dir)
    audio_fast = speed_up_audio(audio, factor=2.0)  # 비용/시간 반으로
    transcript_text = transcribe(audio_fast, settings.openai_api_key)
    save_to_postgres(
        settings.main_database_url,
        persona_id="sosumonkey",  # personas 테이블의 ID
        video_id=info["id"],
        url=info["webpage_url"],
        title=info["title"],
        published_at=datetime.strptime(info["upload_date"], "%Y%m%d"),
        transcript_text=transcript_text,
    )
    print(f"saved: {info['title']} ({len(transcript_text)} chars)")

In [ ]:
from naive_rag.loader import load_transcripts
docs = load_transcripts(limit=3)
print(f"Loaded {len(docs)} documents")
docs[0].metadata, docs[0].page_content[:300]

## Split

In [ ]:
from naive_rag.splitter import split_documents
chunks = split_documents(docs)
print(f"{len(docs)} docs -> {len(chunks)} chunks")
chunks[0].page_content[:200], chunks[0].metadata

## Embed (single query for inspection)

In [ ]:
from naive_rag.embeddings import build_embeddings
emb = build_embeddings()
vec = emb.embed_query("매크로 지표")
print(f"dim={len(vec)}, first 5={vec[:5]}")

## Retrieve

In [ ]:
from naive_rag.retriever import build_retriever
retriever = build_retriever()
hits = retriever.invoke("최근 자주 인용되는 매크로 지표는?")
for i, h in enumerate(hits):
    print(f"--- hit {i} (channel={h.metadata.get('channel')}) ---")
    print(h.page_content[:200])

## Generate

In [ ]:
from naive_rag.chain import build_chain
chain = build_chain()
answer = chain.invoke("최근 트랜스크립트에서 자주 인용되는 매크로 지표는?")
print(answer)

## Run all 10 eval questions

In [ ]:
questions = [
    "최근 AI 투자 전쟁에서 주목한 수혜주는 무엇인가?",
    "AI 에이전트가 반도체 주도주 판도를 어떻게 바꾸고 있나?",
    "트럼프 순방과 젠슨황의 깐부회동이 증시에 미친 영향은?",
    "2026 버블 시나리오에서 본격 버블은 시작됐다고 보나?",
    "최근 폭락한 미국 초우량주 중 매수 후보로 본 종목은?",
    "헬스케어 섹터(유나이티드헬스, 노보노 등)에 대한 시각은?",
    "인텔, AMD, ARM 등 반도체 종목에 대한 평가는?",
    "GTC 워싱턴 행사에서 엔비디아가 발표한 신규 사업 영역은?",
    "2030년까지 컴퓨팅 용량 확장과 AI 인프라 전망은?",
    "빅테크(아마존, 구글, 엔비디아) 중 AI 투자에 가장 적극적인 곳은?",
]
for i, q in enumerate(questions, 1):
    print(f"\n=== Q{i}. {q}")
    print(chain.invoke(q))

## Ragas Evaluation

4 metrics: **Faithfulness**, **AnswerRelevancy**, **LLMContextPrecisionWithoutReference**, **ContextEntityRecall**.

ContextEntityRecall은 reference(정답)에서 entity를 추출해 contexts에 얼마나 들어있는지 본다. 아래 `eval_data` 리스트에서 각 질문의 `reference` 값을 직접 입력. Reference는 핵심 entity(종목명, 인물, 지표명 등)가 들어있는 1-3문장이 좋다.

Judge LLM은 gpt-4o-mini, judge embeddings는 bge-m3.

In [ ]:
from datasets import Dataset
from langchain_openai import ChatOpenAI
from ragas import evaluate
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import (
    AnswerRelevancy,
    ContextEntityRecall,
    Faithfulness,
    LLMContextPrecisionWithoutReference,
)

from naive_rag.chain import build_chain
from naive_rag.config import Settings
from naive_rag.embeddings import build_embeddings
from naive_rag.retriever import build_retriever

# 5개 소수몽키 transcript 내용 기반으로 작성된 reference 정답.
# 직접 수정해서 본인 시각으로 재작성해도 됨.
eval_data = [
    {
        "question": "최근 AI 투자 전쟁에서 주목한 수혜주는 무엇인가?",
        "reference": "구글과 아마존이 엔트로픽에 묻고 더블로 베팅한 빅테크, 자체 반도체(구글 TPU 8세대, 아마존 그래비톤), 그리고 데이터센터 인프라주(GE Vernova 가스터빈, 유나이티드 렌탈·캐터필러 중장비, 이튼·컨스털레이션 전력, 버티브 냉각, 컴포트 시스템 시공)가 수혜주로 언급되었다.",
    },
    {
        "question": "AI 에이전트가 반도체 주도주 판도를 어떻게 바꾸고 있나?",
        "reference": "AI 에이전트 시대에는 다양한 업무를 처리해야 하므로 CPU(인텔, AMD, ARM)의 중요도가 GPU 못지않게 커진다는 평가가 모건스탠리 보고서로 확산되었다. 메타가 아마존의 자체 CPU 그래비톤을 수천만 개 구매한 사건이 이를 뒷받침한다.",
    },
    {
        "question": "트럼프 순방과 젠슨황의 깐부회동이 증시에 미친 영향은?",
        "reference": "트럼프 아시아 순방으로 한미·미일·미중 협력 패키지가 발표되며 미국 주식회사 전체가 투자 유치 수혜를 받았다. 젠슨황의 깐부회동에서 엔비디아는 한국 정부·삼성·현대차·SK·네이버에 반도체 26만 장(약 10-15조원)을 판매했고, GTC 워싱턴 개최로 엔비디아가 국가급 AI 프로젝트의 핵심으로 부상했다.",
    },
    {
        "question": "2026 버블 시나리오에서 본격 버블은 시작됐다고 보나?",
        "reference": "신한투자증권 김성환 연구원의 리포트에 따르면 본격 버블은 2026-2027년에 도래하며 현재는 아직 버블이 아니다. 1995년 다펌 장, 2016년 클라우드 사이클처럼 기술 혁신 주도 사이클은 약 5년 지속되며, PER만으로는 판단이 빡빡해지는 단계라 기술적 분석 병행이 필요하다.",
    },
    {
        "question": "최근 폭락한 미국 초우량주 중 매수 후보로 본 종목은?",
        "reference": "노보노디스크(비만약 위고비, 70% 가까이 폭락)와 유나이티드 헬스케어(미국 최대 민간의료보험, 사상 최악의 폭락)가 한국인 순매수 상위에 오르며 줍줍 대상이 되었다. 다만 보잉처럼 5년 박스권에 갇히는 리스크도 함께 거론되었다.",
    },
    {
        "question": "헬스케어 섹터(유나이티드헬스, 노보노 등)에 대한 시각은?",
        "reference": "트럼프 행정부의 최대 피해주는 제약사와 의료보험사(PBM 중간 유통 마진 타겟). 노보노디스크는 비만약 경쟁 심화로 가이던스 빅배스 의심, 유나이티드 헬스케어는 의료비 폭증으로 실적 쇼크. 다만 1기 때도 약값 누르기 시도가 결국 실패했다는 전설이 있어 회복 사이클 기대도 함께 존재한다.",
    },
    {
        "question": "인텔, AMD, ARM 등 반도체 종목에 대한 평가는?",
        "reference": "인텔은 트럼프 백악관 회동 이후 20불에서 80불로 4배 상승했고 2000년 닷컴버블 고점을 처음 돌파했다. CEO가 'AI 에이전트 시대 CPU 재평가' 구조적 변화를 직접 언급했다. AMD는 CPU 가격 인상과 공급 부족, ARM은 박스권 돌파와 자체 CPU 판매 선언으로 삼총사 모두 신고가를 기록했다.",
    },
    {
        "question": "GTC 워싱턴 행사에서 엔비디아가 발표한 신규 사업 영역은?",
        "reference": "엔비디아는 GTC 워싱턴에서 AI를 미국의 아폴로 프로젝트급으로 선언하고, 양자컴퓨터·슈퍼컴퓨터, 노키아 지분 인수를 통한 6G 통신, 사이버보안, 팔란티어와의 데이터 분석 협업, 자율주행, 일라이 릴리와의 신약 개발, 휴머노이드 로봇 등 전방위 사업 확장을 발표했다.",
    },
    {
        "question": "2030년까지 컴퓨팅 용량 확장과 AI 인프라 전망은?",
        "reference": "OpenAI는 2025년 10GW 목표 중 8GW 확보 완료, 2030년까지 30GW를 새 목표로 발표했다. 반도체·전력주·메모리·광통신이 모두 엮여 있어 AI 인프라 붐은 갈 길이 더 많다(남은 22GW). 현재까지 한 게 8GW이므로 앞으로 할 게 더 많다는 의미.",
    },
    {
        "question": "빅테크(아마존, 구글, 엔비디아) 중 AI 투자에 가장 적극적인 곳은?",
        "reference": "구글이 엔트로픽에 기존 33억불 대비 10배 이상인 400억불(즉시 100억+성과 300억)을 추가 베팅하며 가장 적극적이었고, 아마존도 기존 80억불의 3배 이상인 250억불(즉시 50억+성과 200억)을 추가 투자했다. 엔비디아는 오픈 AI 연합으로 두 진영(엔트로픽 vs 오픈 AI) 경쟁이 핵심 구도가 되었다.",
    },
]

# Reference 미입력 항목 체크
missing = [d["question"] for d in eval_data if not d["reference"].strip()]
if missing:
    raise ValueError(
        f"{len(missing)}개 질문의 reference가 비어 있다. eval_data를 채운 뒤 다시 실행:\n"
        + "\n".join(f"  - {q}" for q in missing)
    )

retriever = build_retriever()
chain = build_chain()
rows = [
    {
        "user_input": d["question"],
        "reference": d["reference"],
        "response": chain.invoke(d["question"]),
        "retrieved_contexts": [c.page_content for c in retriever.invoke(d["question"])],
    }
    for d in eval_data
]

settings = Settings()
result = evaluate(
    Dataset.from_list(rows),
    metrics=[
        Faithfulness(),
        AnswerRelevancy(),
        LLMContextPrecisionWithoutReference(),
        ContextEntityRecall(),
    ],
    llm=LangchainLLMWrapper(
        ChatOpenAI(
            model=settings.llm_model,
            api_key=settings.openai_api_key,
            temperature=0,
        )
    ),
    embeddings=LangchainEmbeddingsWrapper(build_embeddings()),
)
result.to_pandas()